# Wine Quality Classifiers Benchmark

This notebook benchmarks several classical classifiers on the UCI Wine Quality (red) dataset to predict whether a wine is high quality (> 6) or not.

Dependencies
- Python 3.9+
- numpy, pandas
- scikit-learn
- matplotlib, seaborn (for optional insights/plots)

Reproducibility
- Random seed is fixed at 42 for splits and model initialization.
- Set DATA_DIR to override the default data directory (../data relative to this notebook).

Data source and attribution
- Wine Quality Data Set (red wines) by P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis (2009).
- UCI Machine Learning Repository: https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/
- Please review the dataset's terms; this notebook downloads the CSV at runtime if not present locally.

## Dataset

We use the Red Wine subset of the UCI Wine Quality dataset (1599 samples; 11 physicochemical features plus a quality rating from 0–10). The next cell downloads the CSV from the UCI archive into a repository-relative data directory if it is not already present.

In [ ]:
# Core imports and seed
import os
from pathlib import Path
import random
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# Modeling and evaluation imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

## Preprocessing

- Drop any rows with missing values (dataset has none typically, but we enforce this defensively).
- Define the binary target: y = 1 if quality > 6 else 0 (high quality).

In [ ]:
# Download/load the Red Wine Quality dataset (UCI) if not present
import urllib.request

# Repository-relative data directory; can be overridden with DATA_DIR env var
DATA_DIR = Path(os.getenv("DATA_DIR", Path.cwd().parent / "data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using data directory: {DATA_DIR}")
csv_path = DATA_DIR / "winequality-red.csv"

if not csv_path.exists():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
    try:
        print(f"Downloading {url} -> {csv_path}")
        urllib.request.urlretrieve(url, csv_path)
        print("Download complete.")
    except Exception as e:
        raise RuntimeError(
            f"Could not download dataset from {url}. "
            f"Download it manually and place it at {csv_path}, "
            "or set the DATA_DIR environment variable to a writable folder."
        ) from e

# The UCI CSV uses ';' as a delimiter
wine_data = pd.read_csv(csv_path, sep=';')
print(f"Loaded dataset with shape: {wine_data.shape}")

In [ ]:
# Dataset quick facts
print("Samples (rows):", wine_data.shape[0])
print("Features (columns):", wine_data.shape[1])
print("Unique quality ratings:", sorted(wine_data["quality"].unique()))

In [ ]:
# Drop missing values (defensive; UCI file typically has none)
before = len(wine_data)
wine_data = wine_data.dropna().copy()
after = len(wine_data)
print(f"Rows dropped due to missing values: {before - after}")

## Train/Test Split

We create a stratified 70/30 train–test split to preserve class balance, with a fixed seed for reproducibility.

In [ ]:
# Define features and binary target (quality > 6 => 1)
X = wine_data.drop(["quality"], axis=1)
y = (wine_data["quality"] > 6).astype(int).to_numpy()
print(f"Positive class rate (quality > 6): {y.mean():.3f}")

In [ ]:
# Stratified train/test split with fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Test shapes:  X={X_test.shape},  y={y_test.shape}")

## Scaling

Standardize features using StandardScaler (fit on training data, transform on train and test). Tree-based models are scale-invariant, but scaling benefits models like logistic regression and k-NN.

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Scaled arrays: X_train_scaled={X_train_scaled.shape}, X_test_scaled={X_test_scaled.shape}")

## Model Suite

We benchmark a compact suite of common classifiers:
- k-Nearest Neighbors (KNN)
- Decision Tree
- Logistic Regression
- Random Forest

All random states are fixed for reproducibility.

Below we define the models dictionary used in cross-validation and holdout evaluation.

In [ ]:
# Define models (fixed seeds where applicable)
models = {
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=SEED),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED),
}
print("Models:", ", ".join(models.keys()))

## Cross-Validation

We perform stratified 5-fold cross-validation on the training set and report mean accuracy and standard deviation across folds.

In [ ]:
# Stratified 5-fold CV accuracy for each model
k_folds = 5
cv = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=SEED)

for model_name, model in models.items():
    scores = cross_val_score(
        model, X_train_scaled, y_train, cv=cv, scoring="accuracy", n_jobs=-1
    )
    print(f"{model_name:>16} | CV accuracy: mean={scores.mean():.3f}, std={scores.std():.3f} (n={k_folds})")

Notes on model selection:
- Prefer the model with the highest mean CV accuracy and a low standard deviation.
- If scores are similar, prefer the simpler or more interpretable model unless business needs favor otherwise.
- We will evaluate all models on the held-out test set below.

## Holdout Evaluation

Train each model on the full training set (scaled features) before evaluating on the held-out test set.

In [ ]:
# Fit models on the training data
for model_name, model in models.items():
    model.fit(X_train_scaled, y_train)
    print(f"Trained {model_name}")

Evaluate each trained model on the held-out test set. We report accuracy and the full classification report (precision, recall, f1).

In [ ]:
# Test-set performance per model
for model_name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy of {model_name}: {acc:.3f}")
    print(f"Classification report for {model_name}:\n{classification_report(y_test, y_pred, digits=3)}")

## Feature Importance/Insights

Top features by feature_importances_ of a Random Forest trained on the training set. This provides a simple model-based view of which physicochemical properties are most predictive in this dataset.

In [ ]:
# Random Forest feature importances (top 3)
rf_model = RandomForestClassifier(n_estimators=300, random_state=SEED)
rf_model.fit(X_train_scaled, y_train)
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
top3 = importances.head(3)
print("Top 3 features by RandomForest importance:")
print(top3.to_string())

### Additional insights: distributions and correlations
The following optional plots explore feature distributions across quality ratings and the correlation structure.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#### Feature distributions by quality rating

In [ ]:
# Histograms for each feature
for feature in wine_data.columns[:-1]:  # Exclude 'quality' column
    plt.figure(figsize=(8, 6))
    for quality in sorted(wine_data["quality"].unique()):
        sns.histplot(
            wine_data[wine_data["quality"] == quality][feature],
            kde=True,
            label=f"Quality {quality}",
            alpha=0.7,
        )
    plt.title(f"Distribution of {feature} by Quality")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.show()

#### Correlation matrix (features and quality)
Note: Pearson correlations may understate non-linear relationships; treat as directional insight.

In [ ]:
# Correlation Matrix
correlation_matrix = wine_data.corr()
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix, annot=True)
plt.title("Correlation Matrix")
plt.show()